# ماژول های مورد استفاده

In [ ]:
# اتصال گوگل کولب به گوگل درایو
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
# ماژول های مورد استفاده در برنامه
import pandas as pd
import plotly.express as px
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

# وارد کردن داده های فایل CSV و بررسی کلی انها

In [2]:
# خواندن داده ها از فایل csv
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/loans.csv')

# چاپ و نمایش نام ستون ها ، تعداد ستون ها و ردیف ها ، نوع داده ای ستون ها و نمایش تعدادی از نمونه های داده ها
print(f'Data Columns : \n{df.columns}')
print(100*'-')
print(f'Data Shape : {df.shape}')
print(100*'-')
print(f'Data Types : \n{df.dtypes}')
print(100*'-')
df

Data Columns : 
Index(['client_id', 'loan_type', 'loan_amount', 'repaid', 'loan_id',
       'loan_start', 'loan_end', 'rate'],
      dtype='object')
----------------------------------------------------------------------------------------------------
Data Shape : (443, 8)
----------------------------------------------------------------------------------------------------
Data Types : 
client_id        int64
loan_type       object
loan_amount      int64
repaid           int64
loan_id          int64
loan_start      object
loan_end        object
rate           float64
dtype: object
----------------------------------------------------------------------------------------------------


,client_id,loan_type,loan_amount,repaid,loan_id,loan_start,loan_end,rate
0,46109,home,13672,0,10243,2002-04-16,2003-12-20,2.15
1,46109,credit,9794,0,10984,2003-10-21,2005-07-17,1.25
2,46109,home,12734,1,10990,2006-02-01,2007-07-05,0.68
3,46109,cash,12518,1,10596,2010-12-08,2013-05-05,1.24
4,46109,credit,14049,1,11415,2010-07-07,2012-05-21,3.13
...,...,...,...,...,...,...,...,...
438,26945,other,12963,0,10330,2001-11-26,2004-06-11,2.46
439,26945,credit,1728,1,10248,2004-01-27,2005-06-21,5.27
440,26945,other,9329,0,10154,2001-12-17,2004-07-22,5.65
441,26945,home,4197,0,10333,2003-10-16,2005-07-10,4.50


# تشخیص و پردازش مقادیر گم شده

In [3]:
# حذف داده های کاملا خالی
df.dropna(how='all',axis=0)

# تشخیص و نمایش تعداد داده های خالی در هر ستون
df.isna().sum()

# جایگزین کردن داده های خالی با مقدار مد یا میانگین ان ستون
df.fillna(value={
          'client_id' : df['client_id'].mode()[0],
          'loan_type' : df['loan_type'].mode()[0],
          'loan_amount' : df['loan_amount'].mean(),
          'repaid' : df['repaid'].mode()[0],
          'loan_id' : df['loan_id'].mode()[0],
          'loan_start' : df['loan_start'].mode()[0],
          'loan_end' : df['loan_end'].mode()[0],
          'rate' : df['rate'].mean()
          },inplace=True)
df

,client_id,loan_type,loan_amount,repaid,loan_id,loan_start,loan_end,rate
0,46109,home,13672,0,10243,2002-04-16,2003-12-20,2.15
1,46109,credit,9794,0,10984,2003-10-21,2005-07-17,1.25
2,46109,home,12734,1,10990,2006-02-01,2007-07-05,0.68
3,46109,cash,12518,1,10596,2010-12-08,2013-05-05,1.24
4,46109,credit,14049,1,11415,2010-07-07,2012-05-21,3.13
...,...,...,...,...,...,...,...,...
438,26945,other,12963,0,10330,2001-11-26,2004-06-11,2.46
439,26945,credit,1728,1,10248,2004-01-27,2005-06-21,5.27
440,26945,other,9329,0,10154,2001-12-17,2004-07-22,5.65
441,26945,home,4197,0,10333,2003-10-16,2005-07-10,4.50


# تشخیص و پردازش مقادیر پرت

In [4]:
# متد ترسیم جدول جعبه ای برای تشخیص داده های پرت
def check_outliers_data_whit_ploty(data,columns):
  fig = px.box(data,y=columns)
  fig.show()

# ترسیم نمودار و تشخیص داده های پرت در ستون های مقدار وام و نرخ سود
check_outliers_data_whit_ploty(df,['loan_amount'])
check_outliers_data_whit_ploty(df,['rate'])

# حذف داده های پرت موجود در ستون نرخ سود
df = df[df['rate'] <= 10]

# بررسی مجدد داده های ستون نرخ سود
check_outliers_data_whit_ploty(df,['rate'])

# تبدیل مقدار ها


In [5]:
# تبدیل داده های مقدار  وام به داده های لگاریتمی برای یادگیری راحت تر ماشین
df['loan_amount'] = np.log1p(df['loan_amount'])

# تبدیل داده های تاریخی از نوع رشته به تاریخ
df['loan_start'] = pd.to_datetime(df['loan_start'])
df['loan_end'] = pd.to_datetime(df['loan_end'])

# تبدیل و دسته بندی تاریخ ها براساس دو رقم اخر سال ( 0 به معنای 2000 و ...)
df['loan_start'] = pd.qcut(df['loan_start'],q=15,labels=[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14])
df['loan_end'] = pd.qcut(df['loan_end'],q=16,labels=[2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17])
df

/tmp/ipykernel_1247/3144481520.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_1247/3144481520.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_1247/3144481520.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_1247

,client_id,loan_type,loan_amount,repaid,loan_id,loan_start,loan_end,rate
0,46109,home,9.523178,0,10243,2,3,2.15
1,46109,credit,9.189627,0,10984,3,5,1.25
2,46109,home,9.452109,1,10990,6,7,0.68
3,46109,cash,9.435003,1,10596,10,13,1.24
4,46109,credit,9.550378,1,11415,10,12,3.13
...,...,...,...,...,...,...,...,...
438,26945,other,9.469932,0,10330,1,4,2.46
439,26945,credit,7.455298,1,10248,4,5,5.27
440,26945,other,9.140990,0,10154,1,4,5.65
441,26945,home,8.342364,0,10333,3,5,4.50


# رمز گذاری متغییر های دسته ای

In [6]:
# رمز گذاری داده های ستون شناسه مشتری به روش Label Encodeing
le = LabelEncoder()
df['client_id'] = le.fit_transform(df['client_id'])

# رمز گذاری داده های ستون نوع وام به روش One Hot Encodeing
df = pd.get_dummies(df,columns=['loan_type'])
df


/tmp/ipykernel_1247/4145464731.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,client_id,loan_amount,repaid,loan_id,loan_start,loan_end,rate,loan_type_cash,loan_type_credit,loan_type_home,loan_type_other
0,18,9.523178,0,10243,2,3,2.15,False,False,True,False
1,18,9.189627,0,10984,3,5,1.25,False,True,False,False
2,18,9.452109,1,10990,6,7,0.68,False,False,True,False
3,18,9.435003,1,10596,10,13,1.24,True,False,False,False
4,18,9.550378,1,11415,10,12,3.13,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...
438,3,9.469932,0,10330,1,4,2.46,False,False,False,True
439,3,7.455298,1,10248,4,5,5.27,False,True,False,False
440,3,9.140990,0,10154,1,4,5.65,False,False,False,True
441,3,8.342364,0,10333,3,5,4.50,False,False,True,False


# ایجاد متغییر های جدید

In [7]:
# ایجاد متغییر یا ستون جدید به نام باز پرداخت کلی وام که از محاسبه ی ضرب مقدار وام در نرخ سود ان ( به درصد ) و اضافه کردن ان به مقدار وام به دست میاید
# نتیجه محاسبه تبدیل به داده ای لگاریتمی شده است
df['full_loan_repayment'] = np.log1p((np.expm1(df['loan_amount']) * (df['rate'] / 100)) + np.expm1(df['loan_amount']))
df

,client_id,loan_amount,repaid,loan_id,loan_start,loan_end,rate,loan_type_cash,loan_type_credit,loan_type_home,loan_type_other,full_loan_repayment
0,18,9.523178,0,10243,2,3,2.15,False,False,True,False,9.544449
1,18,9.189627,0,10984,3,5,1.25,False,True,False,False,9.202049
2,18,9.452109,1,10990,6,7,0.68,False,False,True,False,9.458886
3,18,9.435003,1,10596,10,13,1.24,True,False,False,False,9.447326
4,18,9.550378,1,11415,10,12,3.13,False,True,False,False,9.581196
...,...,...,...,...,...,...,...,...,...,...,...,...
438,3,9.469932,0,10330,1,4,2.46,False,False,False,True,9.494232
439,3,7.455298,1,10248,4,5,5.27,False,True,False,False,7.506628
440,3,9.140990,0,10154,1,4,5.65,False,False,False,True,9.195946
441,3,8.342364,0,10333,3,5,4.50,False,False,True,False,8.386370


# مقیاس بندی متغییر های عددی

In [8]:
# مقیاس بندی کل داده های عددی به روش نرمالایز کردن
columns = ['client_id','loan_amount','repaid','loan_id','loan_start',
           'loan_end','rate','loan_type_cash','loan_type_credit','loan_type_home',
           'loan_type_other','full_loan_repaymen']
scaler = MinMaxScaler()
data = scaler.fit_transform(df)
df = pd.DataFrame(data)
df.columns = columns
df

,client_id,loan_amount,repaid,loan_id,loan_start,loan_end,rate,loan_type_cash,loan_type_credit,loan_type_home,loan_type_other,full_loan_repaymen
0,0.750,0.972380,0.0,0.118063,0.142857,0.066667,0.216162,0.0,0.0,1.0,0.0,0.961899
1,0.750,0.870873,0.0,0.491927,0.214286,0.200000,0.125253,0.0,1.0,0.0,0.0,0.858195
2,0.750,0.950752,1.0,0.494955,0.428571,0.333333,0.067677,0.0,0.0,1.0,0.0,0.935984
3,0.750,0.945546,1.0,0.296165,0.714286,0.733333,0.124242,1.0,0.0,0.0,0.0,0.932483
4,0.750,0.980658,1.0,0.709384,0.714286,0.666667,0.315152,0.0,1.0,0.0,0.0,0.973029
...,...,...,...,...,...,...,...,...,...,...,...,...
435,0.125,0.956176,0.0,0.161958,0.071429,0.133333,0.247475,0.0,0.0,0.0,1.0,0.946690
436,0.125,0.343080,1.0,0.120585,0.285714,0.200000,0.531313,0.0,1.0,0.0,0.0,0.344699
437,0.125,0.856072,0.0,0.073158,0.071429,0.133333,0.569697,0.0,0.0,0.0,1.0,0.856347
438,0.125,0.613033,0.0,0.163471,0.214286,0.200000,0.453535,0.0,0.0,1.0,0.0,0.611149


# تقسیم داده به مجموعه آموزش و آزمون

In [13]:
# تقسیم داده های آماده شده به داده های مخصوص آزمون و داده های مخصوص ارزیابی
train_data , test_data = train_test_split(df,test_size=0.2,random_state=50)
train_data
test_data


,client_id,loan_amount,repaid,loan_id,loan_start,loan_end,rate,loan_type_cash,loan_type_credit,loan_type_home,loan_type_other,full_loan_repaymen
238,0.291667,0.616063,1.0,0.400101,0.214286,0.266667,0.395960,0.0,0.0,0.0,1.0,0.612508
364,0.083333,0.652353,1.0,0.149344,0.214286,0.266667,0.096970,0.0,1.0,0.0,0.0,0.639876
410,0.541667,0.698270,1.0,0.899596,0.428571,0.466667,0.463636,0.0,1.0,0.0,0.0,0.696271
280,0.250000,0.625949,0.0,0.077699,0.928571,0.933333,0.310101,0.0,0.0,1.0,0.0,0.619860
136,0.208333,0.789629,1.0,0.139253,0.857143,0.866667,0.273737,0.0,1.0,0.0,0.0,0.781703
...,...,...,...,...,...,...,...,...,...,...,...,...
132,0.208333,0.711667,0.0,0.209384,0.714286,0.733333,0.417172,0.0,0.0,0.0,1.0,0.708269
289,0.250000,0.942861,0.0,0.339051,0.571429,0.600000,0.087879,0.0,1.0,0.0,0.0,0.928731
109,0.500000,0.631730,0.0,0.929869,0.500000,0.466667,0.245455,0.0,1.0,0.0,0.0,0.623728
395,0.541667,0.172469,1.0,0.678103,0.928571,0.933333,0.059596,0.0,0.0,0.0,1.0,0.161164
